# 12b — Optimization Algorithms (PyTorch)

Corresponds to: Note 12 — Optimization Algorithms: SGD, Momentum, RMSProp, Adam

**Libraries introduced in this notebook:** No new libraries — `torch` and `torch.nn` introduced in Note 09.

**What this notebook demonstrates:**
1. `torch.optim.SGD` with momentum — verified against 12a
2. `torch.optim.RMSprop` — verified against 12a
3. `torch.optim.Adam` — verified against 12a
4. Learning rate schedules — `StepLR` and `CosineAnnealingLR`
5. Training a network with Adam and a cosine schedule — full training loop pattern

In [ ]:
import torch
import torch.nn as nn

print(f'PyTorch version: {torch.__version__}')

## Part 1: `torch.optim.SGD` with Momentum

In 12a we implemented momentum from scratch. PyTorch provides it via `torch.optim.SGD(params, lr=η, momentum=β)`.

We verify it against the 12a worked example: w₀=1.0, η=0.1, β=0.9, gradients=[0.8, 0.6, 0.7].

Note: PyTorch's SGD momentum uses a slightly different convention:

    vₜ = β · vₜ₋₁ + g          ← no (1−β) scaling
    w  = w − η · vₜ

This is equivalent up to a rescaling of η. We use our own implementation to match the note exactly,
then show the PyTorch version alongside.

In [ ]:
# Our convention (matches Note 12): vₜ = β·vₜ₋₁ + (1−β)·g
grads = [0.8, 0.6, 0.7]
w, v  = 1.0, 0.0
eta, beta = 0.1, 0.9

print('Note 12 convention (matches 12a):')
for t, g in enumerate(grads, 1):
    v = beta * v + (1 - beta) * g
    w = w - eta * v
    print(f'  Step {t}: w = {w:.4f}')

print()

# PyTorch convention: vₜ = β·vₜ₋₁ + g  (no (1−β))
# Equivalent to note convention with η_pytorch = η × (1−β)
w_pt = torch.tensor([1.0], requires_grad=True)
optimizer = torch.optim.SGD([w_pt], lr=eta * (1 - beta), momentum=beta)

print('PyTorch SGD (lr scaled to match note convention):')
for g_val in grads:
    optimizer.zero_grad()
    w_pt.grad = torch.tensor([g_val])
    optimizer.step()
    print(f'  w = {w_pt.item():.4f}')

## Part 2: `torch.optim.RMSprop`

`torch.optim.RMSprop(params, lr=η, alpha=β, eps=ε)` — `alpha` is the smoothing constant (β in the note).

PyTorch's RMSprop matches the note formula exactly. We verify against the 12a worked example.

In [ ]:
w_pt = torch.tensor([1.0], requires_grad=True)
optimizer = torch.optim.RMSprop([w_pt], lr=0.1, alpha=0.9, eps=1e-8)

print('PyTorch RMSprop (α=0.9, lr=0.1):')
for g_val in [0.8, 0.6, 0.7]:
    optimizer.zero_grad()
    w_pt.grad = torch.tensor([g_val])
    optimizer.step()
    print(f'  w = {w_pt.item():.4f}')

print()
print('Expected from 12a: 0.6838, 0.4877, 0.2959')

## Part 3: `torch.optim.Adam`

`torch.optim.Adam(params, lr=η, betas=(β₁, β₂), eps=ε)` — matches the note formula exactly including bias correction.

We verify against the 12a worked example.

In [ ]:
w_pt = torch.tensor([1.0], requires_grad=True)
optimizer = torch.optim.Adam([w_pt], lr=0.1, betas=(0.9, 0.999), eps=1e-8)

print('PyTorch Adam (β₁=0.9, β₂=0.999, lr=0.1):')
for g_val in [0.8, 0.6, 0.7]:
    optimizer.zero_grad()
    w_pt.grad = torch.tensor([g_val])
    optimizer.step()
    print(f'  w = {w_pt.item():.4f}')

print()
print('Expected from 12a: 0.9000, 0.8018, 0.7029')

## Part 4: Learning Rate Schedules

PyTorch provides schedulers in `torch.optim.lr_scheduler`. A scheduler wraps an optimizer and
adjusts its learning rate according to a rule. `scheduler.step()` is called once per epoch.

We demonstrate `StepLR` (step decay) and `CosineAnnealingLR` from Note 12 Part 5.

In [ ]:
# StepLR: reduce lr by factor γ every step_size epochs
w_pt = torch.tensor([1.0], requires_grad=True)
optimizer = torch.optim.SGD([w_pt], lr=0.1)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1)

print('StepLR (step_size=30, gamma=0.1):')
for epoch in range(90):
    scheduler.step()
    if epoch in [0, 29, 30, 59, 60, 89]:
        print(f'  Epoch {epoch+1:>3}: lr = {scheduler.get_last_lr()[0]:.4f}')

print()

In [ ]:
# CosineAnnealingLR: smooth decay from η_max to η_min over T_max epochs
w_pt = torch.tensor([1.0], requires_grad=True)
optimizer = torch.optim.SGD([w_pt], lr=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=90, eta_min=0.001)

print('CosineAnnealingLR (T_max=90, eta_min=0.001):')
for epoch in range(90):
    scheduler.step()
    if epoch in [0, 22, 44, 67, 89]:
        print(f'  Epoch {epoch+1:>3}: lr = {scheduler.get_last_lr()[0]:.4f}')

## Part 5: Full Training Loop with Adam and Cosine Schedule

The complete pattern for training with an optimizer and a learning rate schedule:

    1. Define model, optimizer, scheduler
    2. For each epoch:
       a. For each batch: zero_grad → forward → loss → backward → optimizer.step()
       b. scheduler.step()  ← called once per epoch, after all batches

We train a small network on XOR with Adam and cosine annealing.

In [ ]:
torch.manual_seed(42)

X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])

model = nn.Sequential(
    nn.Linear(2, 8), nn.ReLU(),
    nn.Linear(8, 1), nn.Sigmoid()
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=500, eta_min=1e-4)
criterion = nn.BCELoss()

print(f"{'Epoch':>6}  {'Loss':>8}  {'LR':>10}")
print("-" * 30)
for epoch in range(1, 501):
    optimizer.zero_grad()
    y_hat = model(X)
    loss  = criterion(y_hat, y)
    loss.backward()
    optimizer.step()
    scheduler.step()

    if epoch in [1, 100, 200, 300, 400, 500]:
        lr = scheduler.get_last_lr()[0]
        print(f"{epoch:>6}  {loss.item():>8.4f}  {lr:>10.6f}")

print()
print('Final predictions:')
model.eval()
with torch.no_grad():
    preds = model(X)
    for x_i, p, t in zip(X, preds, y):
        print(f'  x={x_i.tolist()}  y_hat={p.item():.4f}  predicted={int(p.item()>0.5)}  target={int(t.item())}')

## Summary

This notebook demonstrated:

1. **`torch.optim.SGD` with momentum** — PyTorch uses a different momentum convention (no (1−β) scaling). Scaling lr by (1−β) recovers the note formula. Results match 12a.
2. **`torch.optim.RMSprop`** — `alpha` parameter corresponds to β in the note. Matches 12a exactly.
3. **`torch.optim.Adam`** — matches the note formula including bias correction. Matches 12a exactly.
4. **Learning rate schedules** — `StepLR` reduces lr by γ every k epochs (discontinuous). `CosineAnnealingLR` reduces smoothly from η_max to η_min. `scheduler.step()` is called once per epoch.
5. **Full training loop** — optimizer and scheduler used together on XOR. The pattern is: zero_grad → forward → loss → backward → optimizer.step() per batch, then scheduler.step() per epoch.